### Importing the Required Libraries

In [190]:
import pandas as pd
import requests

### Importing the `Solar` + `Wind` Energy Production Data (32 European Countries)

In [45]:
Renew_energy_data = pd.read_csv('Dataset/time_series_60min_singleindex.csv')
Renew_energy_data.head()

,utc_timestamp,cet_cest_timestamp,AT_load_actual_entsoe_transparency,AT_load_forecast_entsoe_transparency,AT_price_day_ahead,AT_solar_generation_actual,AT_wind_onshore_generation_actual,BE_load_actual_entsoe_transparency,BE_load_forecast_entsoe_transparency,BE_solar_generation_actual,...,SI_load_actual_entsoe_transparency,SI_load_forecast_entsoe_transparency,SI_solar_generation_actual,SI_wind_onshore_generation_actual,SK_load_actual_entsoe_transparency,SK_load_forecast_entsoe_transparency,SK_solar_generation_actual,SK_wind_onshore_generation_actual,UA_load_actual_entsoe_transparency,UA_load_forecast_entsoe_transparency
0,2014-12-31T23:00:00Z,2015-01-01T00:00:00+0100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00:00Z,2015-01-01T01:00:00+0100,5946.0,6701.0,35.0,NaN,69.0,9484.0,9897.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01T01:00:00Z,2015-01-01T02:00:00+0100,5726.0,6593.0,45.0,NaN,64.0,9152.0,9521.0,NaN,...,1045.47,816.0,NaN,1.17,2728.0,2860.0,3.8,NaN,NaN,NaN
3,2015-01-01T02:00:00Z,2015-01-01T03:00:00+0100,5347.0,6482.0,41.0,NaN,65.0,8799.0,9135.0,NaN,...,1004.79,805.0,NaN,1.04,2626.0,2810.0,3.8,NaN,NaN,NaN
4,2015-01-01T03:00:00Z,2015-01-01T04:00:00+0100,5249.0,6454.0,38.0,NaN,64.0,8567.0,8909.0,NaN,...,983.79,803.0,NaN,1.61,2618.0,2780.0,3.8,NaN,NaN,NaN


In [ ]:
Renew_energy_data.shape

(50401, 300)

### Filtering the Columns for `Germany` + `United Kingdom`

In [47]:
filtered_columns = [
    'utc_timestamp',
    
    # Germany (DE)
    'DE_load_actual_entsoe_transparency',
    'DE_solar_capacity',
    'DE_solar_generation_actual',
    'DE_wind_generation_actual',
    'DE_wind_onshore_generation_actual',
    'DE_wind_offshore_generation_actual',
    'DE_LU_price_day_ahead',  # price for DE+LU
    
    # United Kingdom (GB)
    'GB_GBN_load_actual_entsoe_transparency',
    'GB_GBN_solar_capacity',
    'GB_GBN_solar_generation_actual',
    'GB_GBN_wind_generation_actual',
    'GB_GBN_wind_onshore_generation_actual',
    'GB_GBN_wind_offshore_generation_actual',
    'GB_GBN_price_day_ahead'
]

In [48]:
filtered_df = Renew_energy_data[filtered_columns]
filtered_df.to_csv('final_filtered_data_DE_GB.csv', index=False)
filtered_df.head()

,utc_timestamp,DE_load_actual_entsoe_transparency,DE_solar_capacity,DE_solar_generation_actual,DE_wind_generation_actual,DE_wind_onshore_generation_actual,DE_wind_offshore_generation_actual,DE_LU_price_day_ahead,GB_GBN_load_actual_entsoe_transparency,GB_GBN_solar_capacity,GB_GBN_solar_generation_actual,GB_GBN_wind_generation_actual,GB_GBN_wind_onshore_generation_actual,GB_GBN_wind_offshore_generation_actual,GB_GBN_price_day_ahead
0,2014-12-31T23:00:00Z,NaN,37248.0,NaN,NaN,NaN,NaN,NaN,NaN,2664.0,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00:00Z,41151.0,37248.0,NaN,8852.0,8336.0,517.0,NaN,26758.0,2669.0,NaN,NaN,NaN,NaN,NaN
2,2015-01-01T01:00:00Z,40135.0,37248.0,NaN,9054.0,8540.0,514.0,NaN,27166.0,2669.0,NaN,782.0,664.0,117.0,NaN
3,2015-01-01T02:00:00Z,39106.0,37248.0,NaN,9070.0,8552.0,518.0,NaN,24472.0,2669.0,NaN,785.0,666.0,120.0,NaN
4,2015-01-01T03:00:00Z,38765.0,37248.0,NaN,9163.0,8643.0,520.0,NaN,23003.0,2669.0,NaN,776.0,662.0,115.0,NaN


In [49]:
filtered_df.shape

(50401, 15)

In [50]:
filtered_df.columns.values

array(['utc_timestamp', 'DE_load_actual_entsoe_transparency',
       'DE_solar_capacity', 'DE_solar_generation_actual',
       'DE_wind_generation_actual', 'DE_wind_onshore_generation_actual',
       'DE_wind_offshore_generation_actual', 'DE_LU_price_day_ahead',
       'GB_GBN_load_actual_entsoe_transparency', 'GB_GBN_solar_capacity',
       'GB_GBN_solar_generation_actual', 'GB_GBN_wind_generation_actual',
       'GB_GBN_wind_onshore_generation_actual',
       'GB_GBN_wind_offshore_generation_actual', 'GB_GBN_price_day_ahead'],
      dtype=object)

### Importing the Weather data for the same Countries and Time period using `Meteo API`

In [36]:
def download_weather_data(latitude, longitude, start_date, end_date, country_code):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,shortwave_radiation,windspeed_10m",
        "timezone": "UTC"
    }
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        weather_df = pd.DataFrame(data['hourly'])
        weather_df['country'] = country_code  # Add country label
        return weather_df
    else:
        print(f"Failed to download weather data for {country_code}. Status code: {response.status_code}")
        return None

In [51]:
start_date = "2014-12-30"
end_date = "2020-09-30"

# Germany coordinates (Berlin roughly)
latitude_de = 52.52
longitude_de = 13.41

# United Kingdom coordinates (London roughly)
latitude_gb = 51.5072
longitude_gb = -0.1276

# Download weather data
weather_Germany = download_weather_data(latitude_de, longitude_de, start_date, end_date, "DE")
weather_UK = download_weather_data(latitude_gb, longitude_gb, start_date, end_date, "GB")

In [54]:
weather_Germany.head()

,time,temperature_2m,shortwave_radiation,windspeed_10m,country
0,2014-12-30T00:00,-5.1,0.0,8.4,DE
1,2014-12-30T01:00,-4.7,0.0,9.2,DE
2,2014-12-30T02:00,-4.2,0.0,9.4,DE
3,2014-12-30T03:00,-4.9,0.0,9.1,DE
4,2014-12-30T04:00,-4.4,0.0,9.7,DE


In [55]:
weather_UK.head()

,time,temperature_2m,shortwave_radiation,windspeed_10m,country
0,2014-12-30T00:00,-2.8,0.0,9.2,GB
1,2014-12-30T01:00,-2.8,0.0,10.0,GB
2,2014-12-30T02:00,-2.9,0.0,10.5,GB
3,2014-12-30T03:00,-3.0,0.0,9.9,GB
4,2014-12-30T04:00,-3.1,0.0,9.4,GB


In [56]:
weather_Germany.to_csv("weather_germany.csv", index=False)
weather_UK.to_csv("weather_uk.csv", index=False)

In [57]:
weather_Germany.columns.values

array(['time', 'temperature_2m', 'shortwave_radiation', 'windspeed_10m',
       'country'], dtype=object)

### Merging the Energy Production Data with the Weather Data

In [163]:
Renew_energy_combined = pd.read_csv('final_filtered_data_DE_GB.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
Renew_energy_combined.head()

,DE_load_actual_entsoe_transparency,DE_solar_capacity,DE_solar_generation_actual,DE_wind_generation_actual,DE_wind_onshore_generation_actual,DE_wind_offshore_generation_actual,DE_LU_price_day_ahead,GB_GBN_load_actual_entsoe_transparency,GB_GBN_solar_capacity,GB_GBN_solar_generation_actual,GB_GBN_wind_generation_actual,GB_GBN_wind_onshore_generation_actual,GB_GBN_wind_offshore_generation_actual,GB_GBN_price_day_ahead
utc_timestamp,,,,,,,,,,,,,,
2014-12-31 23:00:00+00:00,NaN,37248.0,NaN,NaN,NaN,NaN,NaN,NaN,2664.0,NaN,NaN,NaN,NaN,NaN
2015-01-01 00:00:00+00:00,41151.0,37248.0,NaN,8852.0,8336.0,517.0,NaN,26758.0,2669.0,NaN,NaN,NaN,NaN,NaN
2015-01-01 01:00:00+00:00,40135.0,37248.0,NaN,9054.0,8540.0,514.0,NaN,27166.0,2669.0,NaN,782.0,664.0,117.0,NaN
2015-01-01 02:00:00+00:00,39106.0,37248.0,NaN,9070.0,8552.0,518.0,NaN,24472.0,2669.0,NaN,785.0,666.0,120.0,NaN
2015-01-01 03:00:00+00:00,38765.0,37248.0,NaN,9163.0,8643.0,520.0,NaN,23003.0,2669.0,NaN,776.0,662.0,115.0,NaN


In [164]:
weather_Germany = pd.read_csv('weather_germany.csv', parse_dates=['time'], index_col='time')
weather_UK = pd.read_csv('weather_uk.csv', parse_dates=['time'], index_col='time')

#### For `Germany`

In [165]:
energy_germany = Renew_energy_combined[[
    'DE_load_actual_entsoe_transparency',
    'DE_solar_capacity',
    'DE_solar_generation_actual',
    'DE_wind_generation_actual',
    'DE_wind_onshore_generation_actual',
    'DE_wind_offshore_generation_actual',
    'DE_LU_price_day_ahead'
]]
energy_germany.index = energy_germany.index.tz_localize(None)
weather_Germany.index = weather_Germany.index.tz_localize(None)
merged_germany = energy_germany.join(weather_Germany[['temperature_2m', 'shortwave_radiation', 'windspeed_10m']], how='inner')

merged_germany.index.name = 'utc_timestamp'
# merged_germany = merged_germany.reset_index()
print("Germany Merged Data Sample:")
merged_germany.head()

Germany Merged Data Sample:


,DE_load_actual_entsoe_transparency,DE_solar_capacity,DE_solar_generation_actual,DE_wind_generation_actual,DE_wind_onshore_generation_actual,DE_wind_offshore_generation_actual,DE_LU_price_day_ahead,temperature_2m,shortwave_radiation,windspeed_10m
utc_timestamp,,,,,,,,,,
2014-12-31 23:00:00,NaN,37248.0,NaN,NaN,NaN,NaN,NaN,4.0,0.0,14.2
2015-01-01 00:00:00,41151.0,37248.0,NaN,8852.0,8336.0,517.0,NaN,3.8,0.0,14.4
2015-01-01 01:00:00,40135.0,37248.0,NaN,9054.0,8540.0,514.0,NaN,3.6,0.0,14.9
2015-01-01 02:00:00,39106.0,37248.0,NaN,9070.0,8552.0,518.0,NaN,3.3,0.0,14.6
2015-01-01 03:00:00,38765.0,37248.0,NaN,9163.0,8643.0,520.0,NaN,3.0,0.0,14.1


#### For `United Kingdom`

In [166]:
energy_united_kingdom = Renew_energy_combined[[
    'GB_GBN_load_actual_entsoe_transparency',
    'GB_GBN_solar_capacity',
    'GB_GBN_solar_generation_actual',
    'GB_GBN_wind_generation_actual',
    'GB_GBN_wind_onshore_generation_actual',
    'GB_GBN_wind_offshore_generation_actual',
    'GB_GBN_price_day_ahead'
]]

energy_united_kingdom.index = energy_united_kingdom.index.tz_localize(None)
weather_UK.index = weather_UK.index.tz_localize(None)
merged_UK = energy_united_kingdom.join(weather_UK[['temperature_2m', 'shortwave_radiation', 'windspeed_10m']], how='inner')
merged_UK.index.name = 'utc_timestamp'
# merged_UK = merged_UK.reset_index()
print("\nUK Merged Data Sample:")
merged_UK.head()


UK Merged Data Sample:


,GB_GBN_load_actual_entsoe_transparency,GB_GBN_solar_capacity,GB_GBN_solar_generation_actual,GB_GBN_wind_generation_actual,GB_GBN_wind_onshore_generation_actual,GB_GBN_wind_offshore_generation_actual,GB_GBN_price_day_ahead,temperature_2m,shortwave_radiation,windspeed_10m
utc_timestamp,,,,,,,,,,
2014-12-31 23:00:00,NaN,2664.0,NaN,NaN,NaN,NaN,NaN,4.7,0.0,20.3
2015-01-01 00:00:00,26758.0,2669.0,NaN,NaN,NaN,NaN,NaN,4.7,0.0,21.1
2015-01-01 01:00:00,27166.0,2669.0,NaN,782.0,664.0,117.0,NaN,5.0,0.0,20.6
2015-01-01 02:00:00,24472.0,2669.0,NaN,785.0,666.0,120.0,NaN,5.3,0.0,19.9
2015-01-01 03:00:00,23003.0,2669.0,NaN,776.0,662.0,115.0,NaN,5.7,0.0,18.7


### Handling the Missing Values

#### For `Germany`

In [167]:
merged_germany.shape

(50401, 10)

In [168]:
merged_germany.isnull().sum()

DE_load_actual_entsoe_transparency        1
DE_solar_capacity                      6601
DE_solar_generation_actual              104
DE_wind_generation_actual                75
DE_wind_onshore_generation_actual        73
DE_wind_offshore_generation_actual       75
DE_LU_price_day_ahead                 32861
temperature_2m                            0
shortwave_radiation                       0
windspeed_10m                             0
dtype: int64

In [169]:
cleaned_merged_germany = merged_germany.copy()
cleaned_merged_germany.head()

,DE_load_actual_entsoe_transparency,DE_solar_capacity,DE_solar_generation_actual,DE_wind_generation_actual,DE_wind_onshore_generation_actual,DE_wind_offshore_generation_actual,DE_LU_price_day_ahead,temperature_2m,shortwave_radiation,windspeed_10m
utc_timestamp,,,,,,,,,,
2014-12-31 23:00:00,NaN,37248.0,NaN,NaN,NaN,NaN,NaN,4.0,0.0,14.2
2015-01-01 00:00:00,41151.0,37248.0,NaN,8852.0,8336.0,517.0,NaN,3.8,0.0,14.4
2015-01-01 01:00:00,40135.0,37248.0,NaN,9054.0,8540.0,514.0,NaN,3.6,0.0,14.9
2015-01-01 02:00:00,39106.0,37248.0,NaN,9070.0,8552.0,518.0,NaN,3.3,0.0,14.6
2015-01-01 03:00:00,38765.0,37248.0,NaN,9163.0,8643.0,520.0,NaN,3.0,0.0,14.1


##### 1) Forward-filling then back-filling for the column: `DE_load_actual_entsoe_transparency`

In [170]:
cleaned_merged_germany['DE_load_actual_entsoe_transparency'] = cleaned_merged_germany['DE_load_actual_entsoe_transparency'].ffill().bfill()

In [171]:
print("No. of empty rows: ",cleaned_merged_germany['DE_load_actual_entsoe_transparency'].isnull().sum())

No. of empty rows:  0


##### 2) Filling the NaN by constant values for column: `DE_solar_capacity`

In [172]:
solar_cap_const = cleaned_merged_germany['DE_solar_capacity'].dropna().iloc[0]
cleaned_merged_germany['DE_solar_capacity'] = cleaned_merged_germany['DE_solar_capacity'].fillna(solar_cap_const)

In [173]:
print("No. of empty rows: ",cleaned_merged_germany['DE_solar_capacity'].isnull().sum())

No. of empty rows:  0


##### 3) Interpolating values linearly over time and then forward and back filling for the rest of the NaN columns

In [174]:
for col in [
    'DE_solar_generation_actual',
    'DE_wind_generation_actual',
    'DE_wind_onshore_generation_actual',
    'DE_wind_offshore_generation_actual']:
    cleaned_merged_germany[col] = cleaned_merged_germany[col].interpolate(method='time').ffill().bfill()

In [175]:
print("No. of empty rows: ",cleaned_merged_germany['DE_solar_generation_actual'].isnull().sum())

No. of empty rows:  0


##### 4) Dropping the column `DE_LU_price_day_ahead `

In [176]:
cleaned_merged_germany = cleaned_merged_germany.drop(columns=['DE_LU_price_day_ahead'])

In [177]:
cleaned_merged_germany.isnull().sum()

DE_load_actual_entsoe_transparency    0
DE_solar_capacity                     0
DE_solar_generation_actual            0
DE_wind_generation_actual             0
DE_wind_onshore_generation_actual     0
DE_wind_offshore_generation_actual    0
temperature_2m                        0
shortwave_radiation                   0
windspeed_10m                         0
dtype: int64

In [178]:
cleaned_merged_germany = cleaned_merged_germany.reset_index()

In [179]:
cleaned_merged_germany.to_csv("cleaned_germany_energy_weather.csv")

#### For `United Kingdon`

In [180]:
merged_UK.shape

(50401, 10)

In [181]:
print(merged_UK.isnull().sum())

GB_GBN_load_actual_entsoe_transparency       7
GB_GBN_solar_capacity                     6600
GB_GBN_solar_generation_actual              55
GB_GBN_wind_generation_actual               41
GB_GBN_wind_onshore_generation_actual       41
GB_GBN_wind_offshore_generation_actual      41
GB_GBN_price_day_ahead                     111
temperature_2m                               0
shortwave_radiation                          0
windspeed_10m                                0
dtype: int64


In [182]:
cleaned_merged_uk = merged_UK.copy()
cleaned_merged_uk.head()

,GB_GBN_load_actual_entsoe_transparency,GB_GBN_solar_capacity,GB_GBN_solar_generation_actual,GB_GBN_wind_generation_actual,GB_GBN_wind_onshore_generation_actual,GB_GBN_wind_offshore_generation_actual,GB_GBN_price_day_ahead,temperature_2m,shortwave_radiation,windspeed_10m
utc_timestamp,,,,,,,,,,
2014-12-31 23:00:00,NaN,2664.0,NaN,NaN,NaN,NaN,NaN,4.7,0.0,20.3
2015-01-01 00:00:00,26758.0,2669.0,NaN,NaN,NaN,NaN,NaN,4.7,0.0,21.1
2015-01-01 01:00:00,27166.0,2669.0,NaN,782.0,664.0,117.0,NaN,5.0,0.0,20.6
2015-01-01 02:00:00,24472.0,2669.0,NaN,785.0,666.0,120.0,NaN,5.3,0.0,19.9
2015-01-01 03:00:00,23003.0,2669.0,NaN,776.0,662.0,115.0,NaN,5.7,0.0,18.7


##### 1) Forward-filling then back-filling for the column: `GB_GBN_load_actual_entsoe_transparency`

In [183]:
cleaned_merged_uk['GB_GBN_load_actual_entsoe_transparency'] = cleaned_merged_uk['GB_GBN_load_actual_entsoe_transparency'].ffill().bfill()
print("No. of empty rows: ", cleaned_merged_uk['GB_GBN_load_actual_entsoe_transparency'].isnull().sum())

No. of empty rows:  0


##### 2) Filling the NaN by constant values for column: `GB_GBN_solar_capacity`

In [184]:
solar_cap_const_uk = cleaned_merged_uk['GB_GBN_solar_capacity'].dropna().iloc[0]
cleaned_merged_uk['GB_GBN_solar_capacity'] = cleaned_merged_uk['GB_GBN_solar_capacity'].fillna(solar_cap_const_uk)
print("No. of empty rows: ", cleaned_merged_uk['GB_GBN_solar_capacity'].isnull().sum())

No. of empty rows:  0


##### 3) Interpolating values linearly over time and then forward and back filling for the rest of the NaN columns

In [185]:
for col in [
    'GB_GBN_solar_generation_actual',
    'GB_GBN_wind_generation_actual',
    'GB_GBN_wind_onshore_generation_actual',
    'GB_GBN_wind_offshore_generation_actual'
]:
    cleaned_merged_uk[col] = cleaned_merged_uk[col].interpolate(method='time').ffill().bfill()
    
print("No. of empty rows: ", cleaned_merged_uk['GB_GBN_solar_generation_actual'].isnull().sum())

No. of empty rows:  0


##### 4) Dropping the column `GB_GBN_price_day_ahead `

In [186]:
cleaned_merged_uk = cleaned_merged_uk.drop(columns=['GB_GBN_price_day_ahead'])

In [187]:
cleaned_merged_uk.isnull().sum()

GB_GBN_load_actual_entsoe_transparency    0
GB_GBN_solar_capacity                     0
GB_GBN_solar_generation_actual            0
GB_GBN_wind_generation_actual             0
GB_GBN_wind_onshore_generation_actual     0
GB_GBN_wind_offshore_generation_actual    0
temperature_2m                            0
shortwave_radiation                       0
windspeed_10m                             0
dtype: int64

In [188]:
cleaned_merged_uk=cleaned_merged_uk.reset_index()

In [189]:
cleaned_merged_uk.to_csv("cleaned_uk_energy_weather.csv")